# Marzo, 2026
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IreneZMo/Clase-Ibero/blob/main/Clase-Ibero.ipynb)


Hoy vamos a descargar fotos de google street view y analizarlas para obtener:
- Objetos (Modelo YOLOv8)
- Porcentaje de verde
- Textura con gradiente
- Embeddings visuales (Modelo ResNet 18)

## Paso 1: Obtener API
- Generar cuenta en https://console.cloud.google.com/
- Crear API restringida para Google Street View
- Guardar API en notepad en carpeta con nombre "GoogleCloud"

## Paso 2: Obtener PanoID de lugares seleccionados
- Localizar coordenadas geográficas de cinco lugares
- Crear archivo excel con
    - Identificador de la foto: ID
    - Latitud: lat
    - Longitud: lon
- Descargar notebook con funciones para buscar y analizar imagenes de https://github.com/IreneZMo/Clase-Ibero
    - Guardar archivo en la misma carpeta de trabajo
- Empezamos a programar

In [1]:
# programas requeridos
!pip install torch
!pip install opencv-python
!pip install geopandas
!pip install ultralytics
import torch
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import random
import requests
import time
import geopandas as gpd
from tqdm import tqdm
from shapely.geometry import Point, Polygon
from pyproj import CRS, Geod
from math import sqrt
from tqdm import tqdm
from ultralytics import YOLO


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [22]:
#Si estamos trabajando en Google colab, utilzar este snip para anclar drive y usar MyDrive como folder
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [30]:
# Inicialización
### Definimos la carpeta de trabajo
path         =  r"/content/drive/MyDrive/GSV clase"
### Definimos la carpeta en la que se encuentra un archivo llamado "API.txt" que contiene el API: Literal es lo único que tiene el archivo
path_api     =  r"/content/drive/MyDrive/GSV clase/GoogleCloud"

### Definimos la carpeta en la que guardarán las imágenes a descargar
save_folder  =  os.path.join(path, 'street_view_images')

# Crear la carpeta de trabajo si no existe
os.makedirs(path, exist_ok=True)
# Crear la carpeta para la API si no existe
os.makedirs(path_api, exist_ok=True)
# Crear la carpeta para guardar imágenes si no existe
os.makedirs(save_folder, exist_ok=True)

In [31]:
# Leer el API_KEY desde el archivo
with open(os.path.join(path_api, "API_IZM.txt"), "r") as file:
    API_KEY = file.read().strip()  # Usamos .strip() para eliminar cualquier salto de línea u espacio adicional

# Prefijo para resultados
prefijo = "bd_clase" # Es el prefijo que se usará para guardar la información obtenida de las fotos

In [35]:
# Leer la base de datos
#df = pd.read_csv(path + r"/Base_clase.csv") #activar en caso de tener base de datos en csv
df = pd.read_excel(path + r"/Base_clase.xlsx")
df['coordenadas'] = list(zip(df["ID"], df['lat'], df['lon']))
print(df)

   ID        lat        lon                  lugar                 coordenadas
0   1  19.396921 -99.172481                mi casa  (1, 19.396921, -99.172481)
1   2  19.538270 -99.176463    casa de mi tia Rosa   (2, 19.53827, -99.176463)
2   3  19.609734 -99.235410  casa de mi tia Georgi   (3, 19.609734, -99.23541)


In [ ]:
# Cargamos funciones de busqueda y análisis de imagenes
%run ./functions.ipynb
get_nearest_street_view

In [ ]:
# Lista para almacenar los errores
errores_list = []
# Almacenar resultados
fotos_encont = []
# Iterar sobre las coordenadas
for ID, lat, lon in df['coordenadas']:
    print(f"\n🔄 Procesando lugar ID: {ID}...")
    encontrado = None
    # Hacemos una búsqueda para tener la foto más cercana al punto (200 metros a la redonda)
    resultado = get_nearest_street_view(ID, lat, lon, 200, 2)
    if resultado:
        pano_id, real_lat, real_lon = resultado
        encontrado = (ID, lat, lon, real_lat, real_lon, pano_id)
        fotos_encont.append(encontrado)
        print(f"✅ Encontrado para locación {ID} en 200 metros {encontrado} ")
    if not encontrado:
        print(f"❌ Hogar {ID}: No se encontró imagen en 200 metros")
        fotos_encont.append((ID, lat, lon, None, None, None))

In [ ]:
df_resultados = pd.DataFrame(fotos_encont, columns=['ID', 'lat', 'lon', 'real_lat', 'real_lon', 'pano_id'])
print(df_resultados)

In [ ]:
# Leemos el archivo con las observaciones de los hogares
df = df_resultados

In [ ]:
# Nueva columna para registrar el estado de la descarga
df["estado_descarga"] = [("No descargada", "No existe el pano ID")] * len(df)
print(df)

## Paso 3: Descargar imagenes


In [ ]:
# Iterar sobre las filas del DataFrame
for index, row in df.iterrows():
    print(f"\nProcesando observación número: {index}")
    id_hogar = row["ID"]
    pano_id = row["pano_id"]

    if pd.notna(pano_id):  # Si hay un pano_id válido
        estado, mensaje, url = download_street_view_image(pano_id, ID, save_folder)
        df.at[index, "estado_descarga"] = (estado, mensaje)

# Guardar el nuevo archivo con la columna de descarga
df.to_csv(path + r"\resultados_clase.csv", index=False)
print(df)

# Paso 4: Análisis de Imagenes

## YOLO - "You Only Look Once"
Que es YOLO?
Es un algoritmo de detección de objetos basado en CNN. Identifica y cuenta objetos en imágenes.
El algoritmo dibuja 'cajas' alrededor de los objetos y les pone una etiqueta.
https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/train-yolov8-object-detection-on-custom-dataset.ipynb

In [ ]:
# --------------------------------------------------------
# Ejecutar procesamiento de detección de objetos
# --------------------------------------------------------
image_folder = path + r"\street_view_images"

image_files = [os.path.join(image_folder, img) for img in os.listdir(image_folder) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]

# Llamar a la función para procesar las imágenes solo para detección de objetos
df_objetos = procesar_imagenes(image_files)

# Guardar los resultados en un CSV
if not df_objetos.empty:
    df_objetos.to_csv(os.path.join(path, prefijo + "_detecciones_objetos_yolov8.csv"), index=False)
    print("Resultados guardados en detecciones_objetos_yolov8.csv")
else:
    print("No se detectaron objetos en ninguna de las imágenes.")

Que obtenemos? Una aproximación de cuantos objetos de cada tipo hay en la imagen
https://yolov8.com/

In [ ]:
df_objetos

- Son datos estructurados o no estructurados?
- Para la próxima clase: Escoger un estado de la repûlica, sacar 20 fotos y analizarlas con YOLO
- Observan algún patrón en los datos?

## ResNet - "Resudial Network"

It's a **deep CNN** (152 layers!) that extracts **abstract visual features** from images. Think it as a machine that 'understands' images at a very deep level. It doesn't just see 'a car' or 'a building' - it recognizes complex patterns like 'modern architecture,' 'well-maintained surfaces,' 'color harmony,' and many other subtle visual characteristics that are hard to describe in words.

ResNet-152 (Very Deep CNN)            
                                      
   Layer 1: Detect edges                
   Layer 2-10: Detect simple shapes     
   Layer 11-50: Detect object parts     
   Layer 51-100: Detect whole objects   
   Layer 101-152: Detect complex scenes

Trained on ImageNet (14 million images, 1000 categories)

What we get:
A vector of 511 numbers - each number represents an abstract visual feature.

In [ ]:
df_embeddings = procesar_embeddings_batch(image_files, batch_size=50)

df_embeddings.to_csv(os.path.join(path, prefijo + "_embeddings_visuales.csv"), index=False)

print("✅ Proceso terminado. Embeddings guardados en '_embeddings_visuales.csv'")

df_embeddings

## Comparison: YOLO vs ResNet

| Aspect | YOLO | ResNet |
|--------|------|--------|
| **Purpose** | Count specific objects | Extract abstract features |
| **Output** | Numbers of objects (car: 5, person: 2) | 511 abstract numbers |
| **Interpretable?** | ✅ YES (we know what each number means) | ❌ NO (abstract, learned features) |
| **Example** | "maybe 5 cars, maybe 2 people" | "[0.23, 0.91, 0.05, ...]" |


## Porcentaje de verde

In [ ]:
# Procesamos las imágenes para obtener el porcentaje de verde
df_verde = procesar_porcentaje_verde(image_files)

df_verde.to_csv(os.path.join(path, prefijo + "_porcentaje_verde.csv"), index=False)

print("Proceso terminado. El archivo 'porcentaje_verde.csv' ha sido guardado.")

In [ ]:
df_verde

Que obtenemos? Porcentaje de pixeles verdes en una imagen.
Para reflexionar: Que podemos hacer con esta información? que preguntas de investigación se les ocurren esta información sea relevante?

## Reflexión para el martes (Entregar a Isidro)

1. Porque escogiste esos cinco lugares?
2. Que puedes observar en la fotos? que tan parecidas o diferentes son entre ellas?
3. Como puedes interpretar los resultados del modelo YOLO?
4. Crees que el modelo YOLO hizo un buen trabajo comparado a lo que tu estas observando? Si, No, porque?
5. Puedes interpretar los resultados del modelo ResNet? Porque si o porque no? Puedes inferir algo de los resultados de este modelo?